# Phase 4: LLM evaluation with Ollama (gold vs retrieved contexts, LLM-as-judge)

In [ ]:
# Setup: clone the repo and copy the datasets (same layout as retrieval-benchmark-full).
# The Hugging Face token comes from Kaggle Secrets (Add-ons > Secrets > HF_TOKEN); never paste it here.
!pip install -q "datasets<4.0.0" qdrant_client
!git clone https://github.com/Scrowbin/science-document-viewer
!cd science-document-viewer && git checkout feat/app-improvements && git log --oneline -1
!pip install -q -r science-document-viewer/backend/requirements.txt
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["DEBUG"] = "True"  # backend settings need DEBUG or a SECRET_KEY
os.environ["GEMINI_API"] = UserSecretsClient().get_secret("GEMINI_API")  # LLM judge (Gemini API)
!cp -r /kaggle/input/datasets/kristiko/qasper-official /kaggle/working/science-document-viewer/eval/data/
!mv /kaggle/working/science-document-viewer/eval/data/qasper-official /kaggle/working/science-document-viewer/eval/data/qasper_official
!mkdir -p /kaggle/working/science-document-viewer/eval/pdfs
!cp -r /kaggle/input/datasets/kristiko/peerqa /kaggle/working/science-document-viewer/eval/pdfs/peerqa
!find /kaggle/working/science-document-viewer/eval/pdfs/peerqa -name "*.pdf" | wc -l   # expect about 107

In [ ]:
# Install Ollama, then start TWO separate instances - one per GPU on separate ports.
# GPU 0 -> port 11434 (default), GPU 1 -> port 11435.
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, os

# Server on GPU 0 (port 11434)
subprocess.Popen(
    ["ollama", "serve"],
    env=dict(os.environ, CUDA_VISIBLE_DEVICES="0", OLLAMA_HOST="127.0.0.1:11434"),
    stdout=open("/kaggle/working/ollama_gpu0.log", "w"), stderr=subprocess.STDOUT
)

# Server on GPU 1 (port 11435)
subprocess.Popen(
    ["ollama", "serve"],
    env=dict(os.environ, CUDA_VISIBLE_DEVICES="1", OLLAMA_HOST="127.0.0.1:11435"),
    stdout=open("/kaggle/working/ollama_gpu1.log", "w"), stderr=subprocess.STDOUT
)

time.sleep(8)  # wait for both servers to start

# Verify both ports are up
import requests
for port, label in [(11434, 'GPU 0'), (11435, 'GPU 1')]:
    try:
        r = requests.get(f'http://127.0.0.1:{port}/api/tags', timeout=5)
        print(f'Ollama {label} :{port} OK')
    except Exception as e:
        print(f'Ollama {label} :{port} FAILED: {e}')

# Pull all models to both instances
MODELS = ["llama3.2:3b", "qwen2.5:1.5b", "phi4-mini", "qwen3:4b", "gemma3:4b"]
for m in MODELS:
    for port in (11434, 11435):
        r = requests.post(
            f'http://127.0.0.1:{port}/api/pull',
            json={'model': m, 'stream': False}, timeout=3600
        )
        r.raise_for_status()
    print(f'{m}: pulled to both GPUs')


In [ ]:
import os
S = 'science-document-viewer/eval/phases/phase_4_generation/scripts/evaluate_llm_generation.py'
# smoke test on GPU 0
!OLLAMA_BASE_URL=http://127.0.0.1:11434 python {S} --cohort C-RET --condition gold --limit 2 --models llama3.2:3b


1) Validate the judge on the 100 human-labelled answers before trusting it.

In [ ]:
J = 'gemini-flash-latest'  # active Gemini model (or local Ollama model)
!python {S} --judge-validate --judge-model {J}


2) Gold-evidence contexts (all 5 models, both cohorts). Resumable.

In [ ]:
import subprocess, os

p1 = subprocess.Popen(
    ['python', S, '--cohort', 'C-RET', '--condition', 'gold'],
    env=dict(os.environ, CUDA_VISIBLE_DEVICES='0', OLLAMA_BASE_URL='http://127.0.0.1:11434'),
    stdout=open('/kaggle/working/gen_gold_cret.log', 'w'), stderr=subprocess.STDOUT
)
p2 = subprocess.Popen(
    ['python', S, '--cohort', 'PEERQA-PDF', '--condition', 'gold'],
    env=dict(os.environ, CUDA_VISIBLE_DEVICES='1', OLLAMA_BASE_URL='http://127.0.0.1:11435'),
    stdout=open('/kaggle/working/gen_gold_peerqa.log', 'w'), stderr=subprocess.STDOUT
)

try:
    rc1 = p1.wait()
    rc2 = p2.wait()
finally:
    for name, log, rc in [('C-RET', '/kaggle/working/gen_gold_cret.log', rc1 if 'rc1' in dir() else '?'),
                           ('PEERQA-PDF', '/kaggle/working/gen_gold_peerqa.log', rc2 if 'rc2' in dir() else '?')]:
        lines = open(log).readlines()
        print(f'\n=== {name} (exit {rc}) last 20 lines ===')
        print(''.join(lines[-20:]))
    !cd science-document-viewer && zip -qr /kaggle/working/phase4_checkpoint.zip eval/phases/phase_4_generation/artifacts
    print('Checkpoint saved.')


3) Retrieved contexts. Set PIPELINE to the arm chosen from Phase 3 (format `embedder:arm`, e.g. `qwen:convex+bgem3`, `e5:dense`).

In [ ]:
PIPELINE = 'gemma:dense'  # <- Phase 3 choice
import subprocess, os

p1 = subprocess.Popen(
    ['python', S, '--cohort', 'C-RET', '--condition', 'retrieved', '--pipeline', PIPELINE],
    env=dict(os.environ, CUDA_VISIBLE_DEVICES='0', OLLAMA_BASE_URL='http://127.0.0.1:11434'),
    stdout=open('/kaggle/working/gen_ret_cret.log', 'w'), stderr=subprocess.STDOUT
)
p2 = subprocess.Popen(
    ['python', S, '--cohort', 'PEERQA-PDF', '--condition', 'retrieved', '--pipeline', PIPELINE],
    env=dict(os.environ, CUDA_VISIBLE_DEVICES='1', OLLAMA_BASE_URL='http://127.0.0.1:11435'),
    stdout=open('/kaggle/working/gen_ret_peerqa.log', 'w'), stderr=subprocess.STDOUT
)

try:
    rc1 = p1.wait()
    rc2 = p2.wait()
finally:
    for name, log, rc in [('C-RET', '/kaggle/working/gen_ret_cret.log', rc1 if 'rc1' in dir() else '?'),
                           ('PEERQA-PDF', '/kaggle/working/gen_ret_peerqa.log', rc2 if 'rc2' in dir() else '?')]:
        lines = open(log).readlines()
        print(f'\n=== {name} (exit {rc}) last 20 lines ===')
        print(''.join(lines[-20:]))
    !cd science-document-viewer && zip -qr /kaggle/working/phase4_checkpoint.zip eval/phases/phase_4_generation/artifacts
    print('Checkpoint saved.')


4) Judge every answer with Gemini (key from the Kaggle secret GEMINI_API).

In [ ]:
J = 'gemini-flash-latest'  # Gemini model: uses Gemini API (GEMINI_API secret), no GPU needed.
                            # To use a local model instead: set J to an Ollama model name, e.g. 'qwen2.5:1.5b'

import subprocess, os

if J.startswith('gemini'):
    # Gemini API: single process, all cohorts; parallelism won't help (bottleneck is RPD quota, not GPU).
    r = subprocess.run(['python', S, '--judge', '--judge-model', J], env=os.environ)
    print(f'Judge finished (exit {r.returncode})')
else:
    # Local Ollama model: split across both GPUs to run C-RET and PEERQA-PDF in parallel.
    p1 = subprocess.Popen(
        ['python', S, '--judge', '--judge-model', J, '--cohort', 'C-RET'],
        env=dict(os.environ, CUDA_VISIBLE_DEVICES='0', OLLAMA_BASE_URL='http://127.0.0.1:11434'),
        stdout=open('/kaggle/working/judge_cret.log', 'w'), stderr=subprocess.STDOUT
    )
    p2 = subprocess.Popen(
        ['python', S, '--judge', '--judge-model', J, '--cohort', 'PEERQA-PDF'],
        env=dict(os.environ, CUDA_VISIBLE_DEVICES='1', OLLAMA_BASE_URL='http://127.0.0.1:11435'),
        stdout=open('/kaggle/working/judge_peerqa.log', 'w'), stderr=subprocess.STDOUT
    )
    rc1 = p1.wait()
    rc2 = p2.wait()
    for name, log, rc in [('C-RET', '/kaggle/working/judge_cret.log', rc1),
                           ('PEERQA-PDF', '/kaggle/working/judge_peerqa.log', rc2)]:
        lines = open(log).readlines()
        print(f'\n=== {name} (exit {rc}) last 20 lines ===')
        print(''.join(lines[-20:]))

!python {S} --summary-only
print('Judging complete!')


In [ ]:
# Zip the results and download them from the Output tab, then unzip into the repo at the same path.
!cd science-document-viewer && zip -qr /kaggle/working/phase4_artifacts.zip eval/phases/phase_4_generation/artifacts